# 面试问题：SAM 式 Promptable Segmentation 的图像、提示和掩码解码如何协同？

可以直接复述的回答是：promptable segmentation 先编码图像，再把正点、负点或 box 编成空间提示，mask decoder 融合两者输出像素 logits。点提示说明目标/非目标位置，box 提示限制候选区域；同一图像 embedding 可以复用多次交互。提示坐标必须从原图空间严格变换到 encoder 网格，常见故障是 resize 后仍使用原坐标。真实 SAM 使用 ViT 和双向注意力；下面用手写高斯点编码、box map 和像素线性 decoder 保留核心交互，并真实训练。

## 真实案例：六张含两个亮物体的质检图，只分割用户指定目标

每张 8×8 图同时有目标块和干扰块，纯图像阈值会把两者都选中。三张使用正负点，三张使用 box。程序生成图像只用于解释提示机制，不代表真实 SAM 泛化。

In [1]:
import warnings  # 导入告警控制模块
import torch  # 导入 PyTorch 张量与自动微分
warnings.filterwarnings("ignore")  # 隐藏环境告警保持输出清晰
torch.set_num_threads(1)  # 固定 CPU 单线程执行
torch.manual_seed(1402)  # 固定模型初始化
scene_names = ["左上零件", "右下零件", "中心缺陷", "左下包装", "右上标签", "中心按钮"]  # 定义六个分割场景
images = torch.zeros(6, 8, 8)  # 初始化六张灰度图
targets = torch.zeros(6, 8, 8)  # 初始化六张目标掩码
scene_specs = [  # 定义目标矩形、干扰矩形和提示类型
    ((1, 1, 3, 3), (5, 5, 7, 7), "point"),  # 左上点提示目标
    ((5, 5, 7, 7), (1, 1, 3, 3), "point"),  # 右下点提示目标
    ((3, 3, 6, 6), (0, 0, 2, 2), "point"),  # 中心点提示目标
    ((5, 1, 8, 4), (1, 5, 3, 7), "box"),  # 左下 box 目标
    ((1, 5, 4, 8), (5, 0, 7, 2), "box"),  # 右上 box 目标
    ((2, 2, 6, 6), (0, 6, 2, 8), "box"),  # 中心 box 目标
]  # 结束六场景规格
prompts = []  # 收集每张图的点或 box 提示
for scene_index, specification in enumerate(scene_specs):  # 遍历六个场景规格
    target_box, distractor_box, prompt_type = specification  # 解包目标、干扰和提示类型
    top, left, bottom, right = target_box  # 读取目标矩形坐标
    d_top, d_left, d_bottom, d_right = distractor_box  # 读取干扰矩形坐标
    images[scene_index, top:bottom, left:right] = 1.0  # 在图像中绘制目标亮块
    images[scene_index, d_top:d_bottom, d_left:d_right] = 1.0  # 在图像中绘制外观相同干扰块
    targets[scene_index, top:bottom, left:right] = 1.0  # 只把用户指定块写入真值掩码
    positive = ((top + bottom - 1) / 2.0, (left + right - 1) / 2.0) if prompt_type == "point" else None  # 为点提示计算目标中心
    negative = ((d_top + d_bottom - 1) / 2.0, (d_left + d_right - 1) / 2.0) if prompt_type == "point" else None  # 为点提示计算干扰中心
    box = target_box if prompt_type == "box" else None  # 为 box 场景保留目标边界
    prompts.append({"type": prompt_type, "positive": positive, "negative": negative, "box": box})  # 保存当前场景提示
print("输入预览：scene | target_pixels | distractor_pixels | prompt")  # 输出六场景表头
for scene_index, name in enumerate(scene_names):  # 遍历六张提示分割图
    total_bright = int(images[scene_index].sum())  # 统计目标与干扰总亮像素
    target_pixels = int(targets[scene_index].sum())  # 统计目标掩码像素
    print(f"{name:6} | {target_pixels:2d} | {total_bright - target_pixels:2d} | {prompts[scene_index]}")  # 展示提示和双物体结构
print("首图像素：", images[0].int().tolist())  # 展示一张完整八乘八输入

输入预览：scene | target_pixels | distractor_pixels | prompt
左上零件   |  4 |  4 | {'type': 'point', 'positive': (1.5, 1.5), 'negative': (5.5, 5.5), 'box': None}
右下零件   |  4 |  4 | {'type': 'point', 'positive': (5.5, 5.5), 'negative': (1.5, 1.5), 'box': None}
中心缺陷   |  9 |  4 | {'type': 'point', 'positive': (4.0, 4.0), 'negative': (0.5, 0.5), 'box': None}
左下包装   |  9 |  4 | {'type': 'box', 'positive': None, 'negative': None, 'box': (5, 1, 8, 4)}
右上标签   |  9 |  4 | {'type': 'box', 'positive': None, 'negative': None, 'box': (1, 5, 4, 8)}
中心按钮   | 16 |  4 | {'type': 'box', 'positive': None, 'negative': None, 'box': (2, 2, 6, 6)}
首图像素： [[0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 0, 0, 0, 0, 0], [0, 1, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 1, 0], [0, 0, 0, 0, 0, 1, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0]]


## Baseline / 基线：按图像亮度阈值分割

目标和干扰外观相同，阈值基线会同时选择两块。使用 IoU 与同一 target mask 比较。

In [2]:
def mask_iou(prediction, target):  # 手写二值掩码交并比
    intersection = float((prediction * target).sum())  # 计算预测与真值交集像素
    union = float(((prediction + target) > 0).sum())  # 计算预测与真值并集像素
    return intersection / union if union > 0.0 else 1.0  # 返回 IoU 并处理空掩码
baseline_masks = (images > 0.5).float()  # 用亮度阈值同时选中目标和干扰
baseline_ious = [mask_iou(baseline_masks[index], targets[index]) for index in range(6)]  # 计算六场景基线 IoU
print("scene | target_pixels | predicted_pixels | baseline_IoU")  # 输出阈值基线表头
for scene_index, name in enumerate(scene_names):  # 遍历六个场景
    print(f"{name:6} | {int(targets[scene_index].sum()):2d} | {int(baseline_masks[scene_index].sum()):2d} | {baseline_ious[scene_index]:.3f}")  # 展示干扰导致的过分割
baseline_mean_iou = sum(baseline_ious) / len(baseline_ious)  # 计算六图平均基线 IoU
print(f"亮度阈值 mean IoU={baseline_mean_iou:.3f}")  # 汇总同数据基线

scene | target_pixels | predicted_pixels | baseline_IoU
左上零件   |  4 |  8 | 0.500
右下零件   |  4 |  8 | 0.500
中心缺陷   |  9 | 13 | 0.692
左下包装   |  9 | 13 | 0.692
右上标签   |  9 | 13 | 0.692
中心按钮   | 16 | 20 | 0.800
亮度阈值 mean IoU=0.646


## 核心实现：高斯点、负点、Box Map 与 Mask Decoder

每个像素构造 `[image, positive_gaussian, negative_gaussian, inside_box, bias]` 五维特征，decoder 学习融合提示。输出首图正负点特征图和训练梯度。

In [3]:
class PromptMaskDecoder(torch.nn.Module):  # 定义最小提示编码与掩码解码器
    def __init__(self, height=8, width=8):  # 初始化像素网格和可训练融合权重
        super().__init__()  # 初始化 PyTorch 模块基类
        grid_y, grid_x = torch.meshgrid(torch.arange(height), torch.arange(width), indexing="ij")  # 构造八乘八坐标网格
        self.register_buffer("grid_y", grid_y.float())  # 保存不训练的纵坐标网格
        self.register_buffer("grid_x", grid_x.float())  # 保存不训练的横坐标网格
        self.weight = torch.nn.Parameter(torch.tensor([0.5, 0.5, -0.5, 0.5, -1.0]))  # 初始化图像、提示和偏置融合权重
    def point_map(self, point):  # 把点坐标编码为空间高斯图
        if point is None:  # 检查当前提示是否包含该点
            return torch.zeros_like(self.grid_x)  # 无点时返回全零特征图
        point_y, point_x = point  # 读取浮点纵横坐标
        squared_distance = (self.grid_y - point_y) ** 2 + (self.grid_x - point_x) ** 2  # 计算每像素到提示点平方距离
        return torch.exp(-squared_distance / 4.0)  # 使用更宽高斯覆盖三乘三目标并保留空间衰减
    def box_map(self, box):  # 把矩形提示编码为二值空间图
        result = torch.zeros_like(self.grid_x)  # 初始化全零 box 特征
        if box is not None:  # 检查是否存在 box 提示
            top, left, bottom, right = box  # 读取半开区间边界坐标
            result[top:bottom, left:right] = 1.0  # 将 box 内部标为一
        return result  # 返回八乘八 box map
    def forward(self, image, prompt):  # 融合图像和一个结构化提示输出像素 logits
        positive_map = self.point_map(prompt["positive"])  # 编码正点目标区域
        negative_map = self.point_map(prompt["negative"])  # 编码负点干扰区域
        box_feature = self.box_map(prompt["box"])  # 编码 box 内部区域
        bias = torch.ones_like(image)  # 构造像素常数偏置特征
        features = torch.stack([image, positive_map, negative_map, box_feature, bias], dim=-1)  # 形成八乘八乘五提示特征
        logits = features @ self.weight  # 用可训练线性 decoder 生成掩码 logits
        return logits, features  # 返回像素 logits 与完整中间特征
def binary_cross_entropy(logits, target):  # 手写数值稳定像素二元交叉熵
    return (torch.clamp(logits, min=0.0) - logits * target + torch.log1p(torch.exp(-logits.abs()))).mean()  # 对六十四像素求平均损失
prompt_model = PromptMaskDecoder()  # 创建待训练提示分割模型
training_trace = []  # 保存关键步损失、梯度和权重
for step in range(1, 251):  # 对六场景训练两百五十步
    losses = []  # 收集当前步六场景像素损失
    for scene_index in range(6):  # 逐场景执行图像与提示前向
        logits, features = prompt_model(images[scene_index], prompts[scene_index])  # 真实执行 promptable forward
        losses.append(binary_cross_entropy(logits, targets[scene_index]))  # 保存当前掩码监督损失
    loss = torch.stack(losses).mean()  # 计算六场景平均训练目标
    loss.backward()  # 真实执行 backward 得到五维融合梯度
    gradient = prompt_model.weight.grad.detach().clone()  # 保存当前 decoder 梯度
    with torch.no_grad():  # 关闭手写参数更新计算图
        prompt_model.weight -= 0.30 * prompt_model.weight.grad  # 使用 SGD 更新提示融合权重
        prompt_model.weight.grad.zero_()  # 清空本步梯度
    if step in {1, 5, 50, 250}:  # 保存关键收敛节点
        training_trace.append((step, float(loss), gradient.clone(), prompt_model.weight.detach().clone()))  # 记录损失、梯度和权重
with torch.no_grad():  # 进入中间特征观察阶段
    first_logits, first_features = prompt_model(images[0], prompts[0])  # 计算首图提示特征和 logits
print("step | loss | gradient | weight")  # 输出真实训练轨迹表头
for item in training_trace:  # 遍历四个关键训练节点
    print(f"{item[0]:3d} | {item[1]:.5f} | {[round(value, 4) for value in item[2].tolist()]} | {[round(value, 3) for value in item[3].tolist()]}")  # 展示提示特征权重学习
print("首图 positive map：", torch.round(first_features[:, :, 1] * 10.0).div(10.0).tolist())  # 展示正点空间编码
print("首图 negative map：", torch.round(first_features[:, :, 2] * 10.0).div(10.0).tolist())  # 展示负点空间编码

step | loss | gradient | weight
  1 | 0.37320 | [-0.0027, -0.0024, 0.0187, 0.0, 0.2147] | [0.501, 0.501, -0.506, 0.5, -1.064]
  5 | 0.33839 | [-0.0532, -0.0059, 0.0162, -0.0467, 0.1341] | [0.563, 0.507, -0.526, 0.555, -1.239]
 50 | 0.19756 | [-0.0479, -0.0134, 0.0092, -0.0378, 0.0391] | [1.295, 0.664, -0.679, 1.147, -2.133]
250 | 0.08201 | [-0.0134, -0.0106, 0.0056, -0.0127, 0.0137] | [2.801, 1.389, -1.102, 2.407, -3.421]
首图 positive map： [[0.30000001192092896, 0.5, 0.5, 0.30000001192092896, 0.10000000149011612, 0.0, 0.0, 0.0], [0.5, 0.8999999761581421, 0.8999999761581421, 0.5, 0.20000000298023224, 0.0, 0.0, 0.0], [0.5, 0.8999999761581421, 0.8999999761581421, 0.5, 0.20000000298023224, 0.0, 0.0, 0.0], [0.30000001192092896, 0.5, 0.5, 0.30000001192092896, 0.10000000149011612, 0.0, 0.0, 0.0], [0.10000000149011612, 0.20000000298023224, 0.20000000298023224, 0.10000000149011612, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0,

## 六图结果表：同一图像由提示选择目标

In [4]:
predicted_masks = []  # 收集六场景提示模型二值掩码
prompt_ious = []  # 收集六场景提示模型 IoU
print("scene | prompt_type | baseline_IoU | prompt_IoU | predicted_pixels")  # 输出逐场景对照表头
with torch.no_grad():  # 关闭六场景推理梯度记录
    for scene_index, name in enumerate(scene_names):  # 遍历六张质检图
        logits, features = prompt_model(images[scene_index], prompts[scene_index])  # 复用同一图像提示解码器
        prediction = (torch.sigmoid(logits) >= 0.5).float()  # 将像素概率阈值化为掩码
        iou = mask_iou(prediction, targets[scene_index])  # 计算指定目标 IoU
        predicted_masks.append(prediction)  # 保存当前预测掩码
        prompt_ious.append(iou)  # 保存当前场景指标
        print(f"{name:6} | {prompts[scene_index]['type']:5} | {baseline_ious[scene_index]:.3f} | {iou:.3f} | {int(prediction.sum())}")  # 展示提示带来的目标选择
prompt_mean_iou = sum(prompt_ious) / len(prompt_ious)  # 计算六图平均提示 IoU
print(f"mean IoU：baseline={baseline_mean_iou:.3f}，promptable={prompt_mean_iou:.3f}")  # 汇总同数据改进
print("首图预测 mask：", predicted_masks[0].int().tolist())  # 展示真实二值结果矩阵

scene | prompt_type | baseline_IoU | prompt_IoU | predicted_pixels
左上零件   | point | 0.500 | 1.000 | 4
右下零件   | point | 0.500 | 1.000 | 4
中心缺陷   | point | 0.692 | 1.000 | 9
左下包装   | box   | 0.692 | 1.000 | 9
右上标签   | box   | 0.692 | 1.000 | 9
中心按钮   | box   | 0.800 | 1.000 | 16
mean IoU：baseline=0.646，promptable=1.000
首图预测 mask： [[0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 0, 0, 0, 0, 0], [0, 1, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0]]


## 失败案例与修正：原图坐标未缩放到 encoder 网格

假设首图从 16×16 缩放到 8×8，原图正负点坐标必须除以 2。错误地把原坐标直接传给 8×8 prompt encoder，会让高斯中心偏移甚至落到图外。

In [5]:
correct_prompt = prompts[0]  # 读取首图八乘八正确点提示
original_positive = tuple(value * 2.0 for value in correct_prompt["positive"])  # 构造对应十六乘十六原图正点坐标
original_negative = tuple(value * 2.0 for value in correct_prompt["negative"])  # 构造对应十六乘十六原图负点坐标
wrong_prompt = {"type": "point", "positive": original_positive, "negative": original_negative, "box": None}  # 复现未做坐标缩放的错误提示
fixed_prompt = {"type": "point", "positive": tuple(value / 2.0 for value in original_positive), "negative": tuple(value / 2.0 for value in original_negative), "box": None}  # 将原图点除以缩放因子
with torch.no_grad():  # 关闭坐标失败对照的梯度记录
    wrong_logits, wrong_features = prompt_model(images[0], wrong_prompt)  # 用错误坐标执行掩码解码
    fixed_logits, fixed_features = prompt_model(images[0], fixed_prompt)  # 用变换后坐标执行掩码解码
wrong_mask = (torch.sigmoid(wrong_logits) >= 0.5).float()  # 阈值化错误坐标预测
fixed_mask = (torch.sigmoid(fixed_logits) >= 0.5).float()  # 阈值化正确坐标预测
wrong_iou = mask_iou(wrong_mask, targets[0])  # 计算坐标错误 IoU
fixed_iou = mask_iou(fixed_mask, targets[0])  # 计算坐标修正 IoU
print("原图坐标：", original_positive, original_negative)  # 展示待变换提示
print("错误直接使用 IoU：", round(wrong_iou, 3), "正点峰值位置：", torch.nonzero(wrong_features[:, :, 1] == wrong_features[:, :, 1].max()).tolist())  # 展示高斯提示偏移
print("除以2后 IoU：", round(fixed_iou, 3), "正点峰值位置：", torch.nonzero(fixed_features[:, :, 1] == fixed_features[:, :, 1].max()).tolist())  # 展示坐标对齐修正

原图坐标： (3.0, 3.0) (11.0, 11.0)
错误直接使用 IoU： 0.25 正点峰值位置： [[3, 3]]
除以2后 IoU： 1.0 正点峰值位置： [[1, 1], [1, 2], [2, 1], [2, 2]]


## 结果解读

亮度基线无法区分外观相同的目标和干扰；正负点或 box 增加了“用户指谁”的信息。训练轨迹显示 decoder 学会正点/box 正权重和负点负权重。坐标失败说明 prompt encoder 的输入契约必须包含原图尺寸、resize 和 padding 变换。

## 生产边界

教学模型是五特征像素分类器，没有 SAM 的 image encoder、token attention、多 mask 输出或 IoU head。生产系统应缓存图像 embedding，支持点/box/mask prompt 迭代，并在不同分辨率和 padding 下单测坐标。评估需覆盖小目标、边界质量、空提示和对抗点击，不能用六张程序图外推。

## 最小回归测试

In [6]:
assert len(images) >= 6 and len(prompts) == 6  # 保证提示分割案例包含六个场景
assert set(prompt["type"] for prompt in prompts) == {"point", "box"}  # 保证点和框两类提示均实际覆盖
assert training_trace[-1][1] < training_trace[0][1]  # 保证真实 backward 更新降低像素损失
assert prompt_mean_iou > baseline_mean_iou  # 保证提示模型优于同图亮度基线
assert all(iou >= 0.75 for iou in prompt_ious)  # 保证六场景均能选择指定目标
assert wrong_iou < fixed_iou  # 保证坐标未缩放失败及修正真实可复现
assert fixed_iou == prompt_ious[0]  # 保证正确坐标变换恢复首图标准结果